# Experiment 2: Architecture Change — Widen + Residual

**Single variable changed**: CNN architecture only. Everything else identical to Experiment 1.

## Changes from DiagnosticCNN (E1)

| Property | E1 (DiagnosticCNN) | E2 (ResidualCNN) |
|----------|:------------------:|:-----------------:|
| Conv widths | 32 → 64 → 128 | **64 → 128 → 256** |
| Residual connections | None | **Conv1×1 skip from block 1 → block 3** |
| Activation | ReLU | **LeakyReLU(0.1)** |
| Dropout | 0.3 | **0.5** |
| Everything else | — | Identical |

**Hypothesis**: Wider conv filters capture more subtle boundary cues for the upper-body cluster; residual connections stabilise gradient flow to early layers; LeakyReLU prevents dead neurons in crowded decision regions; higher Dropout reduces the "Shirt sink" over-prediction bias.

In [5]:
import sys
sys.path.append('..')

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt

from src.data_utils import get_fashionmnist_transforms, load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)

OUT_DIR = '../outputs/error_analysis/arch_widen_residual'
os.makedirs(OUT_DIR, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'
print(f"Device: {device}")

PyTorch: 2.13.0+cu130
Device: cuda


## Dataset — identical to E1

In [6]:
transform = get_fashionmnist_transforms()
train_dataset, test_dataset = load_fashionmnist(transform)
class_names = train_dataset.classes
train_loader, test_loader = get_dataloaders(train_dataset, test_dataset, batch_size=64)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 938, Test batches: 157


## Architecture — single variable change

Only this cell differs from Experiment 1. All training hyperparameters, data pipeline, and evaluation code remain unchanged.

In [7]:
class ResidualCNN(nn.Module):
    """Wider conv blocks + residual skip connection + LeakyReLU + Dropout 0.5."""
    def __init__(self, num_classes=10):
        super().__init__()
        # Block 1: 1 -> 64
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2)

        # Block 2: 64 -> 128
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2)

        # Block 3: 128 -> 256 (wider than E1's 128)
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(256)
        self.conv6 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn6 = nn.BatchNorm2d(256)

        # Residual skip: 64 -> 256 via 1x1 conv + pool
        self.residual = nn.Sequential(
            nn.Conv2d(64, 256, kernel_size=1),
            nn.MaxPool2d(4),   # 14 -> 3 (matches block 3 output spatial dim)
        )

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256, num_classes)
        self.act = nn.LeakyReLU(negative_slope=0.1, inplace=True)

    def get_features(self, x):
        # Block 1 (save for residual)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.act(self.bn2(self.conv2(x)))
        x = self.pool1(x)                # 28x28 -> 14x14
        skip = self.residual(x)          # 64ch -> 256ch, 14x14 -> 3x3 via MaxPool2d(4)

        # Block 2
        x = self.act(self.bn3(self.conv3(x)))
        x = self.act(self.bn4(self.conv4(x)))
        x = self.pool2(x)                # 14x14 -> 7x7

        # Block 3 (wider)
        x = self.act(self.bn5(self.conv5(x)))
        x = self.act(self.bn6(self.conv6(x)))
        x = nn.functional.max_pool2d(x, 2)  # 7x7 -> 3x3

        x = x + skip                      # residual addition
        x = self.global_pool(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        x = self.get_features(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = ResidualCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"ResidualCNN params: {total_params:,}")

# Verify no attention
for name, module in model.named_modules():
    cls = module.__class__.__name__
    if 'Attention' in cls or 'SE' in cls or 'CBAM' in cls or 'Transformer' in cls:
        raise RuntimeError(f"Attention found: {name} -> {cls}")
print("Validation: no attention mechanisms present.")

ResidualCNN params: 1,165,258
Validation: no attention mechanisms present.


## Training — identical hyperparameters to E1

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 15

train_losses = []
model.train()
for epoch in range(num_epochs):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')
print(f"Losses saved to {OUT_DIR}/train_losses.txt")

Epoch [1/15], Loss: 0.4440
Epoch [2/15], Loss: 0.2720
Epoch [3/15], Loss: 0.2298
Epoch [4/15], Loss: 0.2027
Epoch [5/15], Loss: 0.1847
Epoch [6/15], Loss: 0.1646
Epoch [7/15], Loss: 0.1480
Epoch [8/15], Loss: 0.1309
Epoch [9/15], Loss: 0.1159
Epoch [10/15], Loss: 0.0981
Epoch [11/15], Loss: 0.0853
Epoch [12/15], Loss: 0.0750
Epoch [13/15], Loss: 0.0616
Epoch [14/15], Loss: 0.0525
Epoch [15/15], Loss: 0.0474
Losses saved to ../outputs/error_analysis/arch_widen_residual/train_losses.txt


## Evaluation — identical to E1

In [9]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='ResidualCNN')
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='ResidualCNN')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='ResidualCNN')

# Single consolidated metrics file
with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write('-' * 55 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')

# Raw confusion matrix
cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names:
        f.write(f'{name:>15}')
    f.write('\n')
    for i in range(len(class_names)):
        f.write(f'{class_names[i]:>15}')
        for j in range(len(class_names)):
            f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

# Identify confusion pairs from the matrix
with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        true_name = class_names[c]
        total_errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {true_name}  (errors: {total_errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0:
                continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f"\nAll results saved to {OUT_DIR}/")

  Test Accuracy: 92.65%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.9390     0.0233     0.8172
  Trouser             0.9890     0.0004     0.9960
  Pullover            0.9310     0.0193     0.8425
  Dress               0.9280     0.0069     0.9374
  Coat                0.8280     0.0063     0.9356
  Sandal              0.9780     0.0008     0.9929
  Shirt               0.7320     0.0171     0.8262
  Sneaker             0.9880     0.0050     0.9564
  Bag                 0.9840     0.0007     0.9939
  Ankle boot          0.9680     0.0018     0.9837

All results saved to ../outputs/error_analysis/arch_widen_residual/


## Delta Report — vs Experiment 1 Baseline

The table below will be populated after execution by running the comparison script.

In [10]:
import json

# Load E1 baseline metrics
def load_metrics(path):
    data = {}
    with open(path) as f:
        for line in f:
            if ':' in line and '---' not in line and 'Class' not in line:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, roc, pr, tpr, prec = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    data[cls] = {'roc_auc': roc, 'pr_auc': pr, 'tpr': tpr, 'precision': prec}
    return data

e1 = load_metrics('../outputs/error_analysis/metrics_summary.txt')

print(f'{"Class":<15} {"E1 Acc":>8} {"E2 Acc":>8} {"Δ Acc":>8} {"E1 PR":>8} {"E2 PR":>8} {"Δ PR":>8}')
print('-' * 63)
for name in class_names:
    if name in e1:
        e1_tpr = e1[name]['tpr']
        e1_pr = e1[name]['pr_auc']
        e2_tpr = per_class[name]['TPR']
        e2_pr = pr_scores[f'class_{class_names.index(name)}']
        print(f'{name:<15} {e1_tpr:>8.3f} {e2_tpr:>8.3f} {e2_tpr - e1_tpr:>+8.3f} {e1_pr:>8.3f} {e2_pr:>8.3f} {e2_pr - e1_pr:>+8.3f}')

print(f'\nAccuracy:  E1=92.50%  E2={accuracy:.2f}%  Δ={accuracy - 92.50:+.2f}%')
print(f'Macro PR:   E1=0.9712  E2={pr_scores["macro"]:.4f}  Δ={pr_scores["macro"] - 0.9712:+.4f}')

Class             E1 Acc   E2 Acc    Δ Acc    E1 PR    E2 PR     Δ PR
---------------------------------------------------------------

Accuracy:  E1=92.50%  E2=92.65%  Δ=+0.15%
Macro PR:   E1=0.9712  E2=0.9713  Δ=+0.0001


## Results saved to `outputs/error_analysis/arch_widen_residual/`

| File | Contents |
|------|----------|
| `train_losses.txt` | Per-epoch training loss |
| `metrics_summary.txt` | Accuracy, per-class ROC-AUC, PR-AUC, TPR, Precision |
| `confusion_matrix.txt` | Raw confusion matrix |
| `misclassification_analysis.txt` | Per-class error breakdown |